In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
from sqlalchemy import create_engine
print("All libraries lodaded")

All libraries lodaded


In [2]:
nav_df = pd.read_csv("data/raw/02_nav_history.csv")
fund_df = pd.read_csv("data/raw/01_fund_master.csv")

print("nav_history shape:", nav_df.shape)
print("fund_master shape:", fund_df.shape)
print("\nNAV columns:", nav_df.columns.tolist())
print("Fund columns:", fund_df.columns.tolist())

nav_history shape: (46000, 3)
fund_master shape: (40, 15)

NAV columns: ['amfi_code', 'date', 'nav']
Fund columns: ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']


In [3]:
print("=== NAV HISTORY ===")
print(nav_df.head())

print("\n=== FUND MASTER ===")
print(fund_df.head())

=== NAV HISTORY ===
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692

=== FUND MASTER ===
   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular Plan - Growth   

  category sub_category     plan launch_date                  benchmark  \
0   Equity    Large Cap  Regular  2006-02-14              NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01              NIFTY 100 TRI   
2   Equity    Small Cap  Regular  2009-09-09       BSE 

In [4]:
print("=== NAV NULL VALUES ===")
print(nav_df.isnull().sum())

print("\n=== FUND NULL VALUES ===")
print(fund_df.isnull().sum())

print("\n=== NAV DATA TYPES ===")
print(nav_df.dtypes)

=== NAV NULL VALUES ===
amfi_code    0
date         0
nav          0
dtype: int64

=== FUND NULL VALUES ===
amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

=== NAV DATA TYPES ===
amfi_code      int64
date          object
nav          float64
dtype: object


In [5]:
nav_df.columns=nav_df.columns.str.lower().str.strip()
nav_df["date"]=pd.to_datetime(nav_df["date"],dayfirst=True,errors="coerce")
nav_df["nav"]=pd.to_numeric(nav_df["nav"],errors="coerce")
nav_df=nav_df.sort_values(["amfi_code","date"])
nav_df=nav_df.drop_duplicates()
nav_df["nav"]=nav_df.groupby("amfi_code")["nav"].ffill()
nav_df=nav_df[nav_df["nav"]>0]

print("NAV Cleaned!")
print("Shape:",nav_df.shape)
print("Null values:\n",nav_df.isnull().sum())

NAV Cleaned!
Shape: (45962, 3)
Null values:
 amfi_code        0
date         27842
nav              0
dtype: int64


In [7]:
fund_df.columns=fund_df.columns.str.lower().str.strip()
fund_df["fund_manager"]=fund_df["fund_manager"].fillna("Unknown")
fund_df["benchmark"]=fund_df["benchmark"].fillna("Unknown")
fund_df["expense_ratio_pct"]=pd.to_numeric(fund_df["expense_ratio_pct"],errors="coerce")
fund_df["min_sip_amount"]=pd.to_numeric(fund_df["min_sip_amount"],errors="coerce")
fund_df["min_lumpsum_amount"]=pd.to_numeric(fund_df["min_lumpsum_amount"],errors="coerce")

fund_df=fund_df[
    (fund_df["expense_ratio_pct"]>=0.1)&
    (fund_df["expense_ratio_pct"]<=2.5)
    ]
fund_df["launch_date"]=pd.to_datetime(fund_df["launch_date"],errors="coerce")

print("Fund master cleaned")
print("Shape:",fund_df.shape)
print("Null values:\n",fund_df.isnull().sum())


Fund master cleaned
Shape: (40, 15)
Null values:
 amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64


In [9]:
os.makedirs("data/processed",exist_ok=True)
nav_df.to_csv("data/processed/nav_history_cleaned.csv",index=False)
fund_df.to_csv("data/processed/fund_master_cleaned.csv",index=False)
print("Cleaned files saved to data/processed/")

Cleaned files saved to data/processed/


In [10]:
os.makedirs("data/db",exist_ok=True)
engine=create_engine("sqlite:///data/db/bluestock_mf.db")
nav_df.to_sql("fact_nav",engine,if_exists="replace",index=False)
fund_df.to_sql("dim_fund",engine,if_exists="replace",index=False)

print("Database created")
conn=sqlite3.connect("data/db/bluestock_mf.db")
print("fact_nav rows:",pd.read_sql("SELECT COUNT(*) as count FROM fact_nav",conn).iloc[0,0])
print("dim_fund rows:",pd.read_sql("SELECT COUNT(*) AS count FROM dim_fund",conn).iloc[0,0])
conn.close()

Database created
fact_nav rows: 45962
dim_fund rows: 40
